# Semana 9 · Sesión 2: Espín y oscilador armónico

**Módulo 2**

## Objetivos de la sesión

1. Trabajar con espín y momento angular en `quantum.spin`: aplicar operadores
   con `qapply`, obtener matrices con `represent` y combinar espines con
   `TensorProduct`.
2. Resolver el oscilador armónico con operadores de escalera, sin escribir
   ninguna función de onda.

## Retomamos

La sesión pasada los operadores eran abstractos: `A`, `B`, `C`, sin
representación. Hoy son concretos: $J_x, J_y, J_z$ actúan sobre estados
$|j, m\rangle$ y `qapply` calcula el resultado.

Esta celda es el único preparativo: no depende de la sesión anterior.

In [ ]:
import sympy as sp
from IPython.display import Math
from sympy.physics.matrices import msigma
from sympy.physics.quantum import Commutator, Dagger, TensorProduct, qapply, represent
from sympy.physics.quantum.constants import hbar
from sympy.physics.quantum.sho1d import (
    Hamiltonian, LoweringOp, NumberOp, RaisingOp, SHOBra, SHOKet,
)
from sympy.physics.quantum.spin import J2, Jminus, Jplus, Jx, Jy, Jz, JzKet

sp.init_printing()

## Espín: estados y operadores

`JzKet(j, m)` es el estado $|j, m\rangle$. Los operadores `Jx`, `Jy`, `Jz`,
`J2`, `Jplus` y `Jminus` ya traen sus reglas, con $\hbar$ como constante (`hbar`, de `quantum.constants`), y
`qapply` las aplica. Para $j = 1$:

In [ ]:
estado = JzKet(1, 0)

display(sp.Eq(Jz * estado, qapply(Jz * estado), evaluate=False))
display(sp.Eq(J2 * estado, qapply(J2 * estado), evaluate=False))
display(sp.Eq(Jplus * estado, qapply(Jplus * estado), evaluate=False))

Observa que $J_+$ sube $m$, que $J_z|1,0\rangle = 0$ y que
$J^2 = \hbar^2 j(j+1) = 2\hbar^2$. Los operadores de Cartesianas cumplen el
álgebra de momento angular, que `Commutator` reproduce sola:

In [ ]:
display(sp.Eq(Commutator(Jx, Jy), Commutator(Jx, Jy).doit(), evaluate=False))
display(sp.Eq(Commutator(J2, Jz), Commutator(J2, Jz).doit(), evaluate=False))

## De operadores a matrices: `represent`

`represent(operador, basis=Jz, j=...)` da la matriz del operador en la base
$|j, m\rangle$. Para $j = 1/2$ salen las matrices de Pauli por $\hbar/2$:

In [ ]:
semi = sp.Rational(1, 2)

matriz_jx = represent(Jx, basis=Jz, j=semi)
matriz_jy = represent(Jy, basis=Jz, j=semi)

display(Math(r"J_x = " + sp.latex(matriz_jx)))   # una matriz no entra en sp.Eq: Eq la daría por distinta de un símbolo
display(Math(r"J_y = " + sp.latex(matriz_jy)))
display(Math(r"\sigma_x = " + sp.latex(msigma(1))))

## TODO en clase 1

El **espín 1**. Con $j = 1$ las matrices son $3\times 3$.

1. Calcula `jx_1`, `jy_1` y `jz_1` con `represent(..., basis=Jz, j=1)`.
2. Comprueba el álgebra con matrices: `jx_1 * jy_1 - jy_1 * jx_1` debe ser
   $i\hbar J_z$ (resta `sp.I * hbar * jz_1` y simplifica a la matriz cero).
3. Obtén los eigenvalores de `jx_1` con `eigenvals()`. ¿Por qué son los mismos
   que los de $J_z$?
4. Aplica `Jminus` dos veces al estado `JzKet(1, 1)` con `qapply` y guárdalo
   en `estado_abajo`.

In [ ]:
# TODO en clase: el espín 1
jx_1 = ...
jy_1 = ...
jz_1 = ...
conmutador_matrices = ...
valores_propios = ...
estado_abajo = ...

## Dos espines: `TensorProduct`

Para dos partículas de espín 1/2 el espacio es el producto tensorial. Con
matrices se hace con `TensorProduct`. El hamiltoniano de Heisenberg
$H = \vec\sigma_1\cdot\vec\sigma_2$ separa el estado **singlete** (el de
espín total 0) de los tres estados del **triplete**:

In [ ]:
sx, sy, sz = [sp.Matrix(msigma(k)) for k in (1, 2, 3)]

heisenberg = (
    TensorProduct(sx, sx) + TensorProduct(sy, sy) + TensorProduct(sz, sz)
)

display(Math(r"H = " + sp.latex(heisenberg)))
display(heisenberg.eigenvals())

El triplete tiene energía $+1$ (tres veces) y el singlete $-3$. Los
eigenvalores de $\vec\sigma_1\cdot\vec\sigma_2$ son $2S(S+1) - 3$ con $S$ el
espín total, así que $S = 1 \to +1$ y $S = 0 \to -3$. Esa fórmula la tomamos prestada:
ahora la comprobamos.

## TODO en clase 2

El **espín total** de dos espines 1/2. Para cada partícula, $\vec S = \frac{\hbar}{2}\vec\sigma$; el
espín total suma el de la primera (que actúa sobre el primer factor) y el de la
segunda: $S_k = s_k \otimes \mathbb{1} + \mathbb{1} \otimes s_k$.

1. Completa `espin_total` con las tres componentes $S_x, S_y, S_z$, usando
   `TensorProduct(s_k, identidad) + TensorProduct(identidad, s_k)`.
2. Calcula `espin_total_cuadrado`, la matriz $S^2 = S_x^2 + S_y^2 + S_z^2$
   (de $4\times 4$). Ojo: `sum()` de Python empieza a sumar desde el entero
   `0`, que no se suma con una matriz; si lo usas, dale `sp.zeros(4)` como
   segundo argumento.
3. Obtén sus eigenvalores con `eigenvals()` y guárdalos en
   `valores_propios_total`. Deben ser $\hbar^2 S(S+1)$: ¿cuántas veces sale
   $S = 0$ y cuántas $S = 1$?
4. Multiplica $S^2$ por el `singlete` y simplifica: ¿qué esperas obtener?
5. ¿Cómo se relaciona esto con la fórmula $2S(S+1) - 3$ de arriba?

In [ ]:
# TODO en clase: el espín total
s = [sp.Matrix(msigma(k)) * hbar / 2 for k in (1, 2, 3)]   # el espín de una partícula
identidad = sp.eye(2)
singlete = sp.Matrix([0, 1, -1, 0]) / sp.sqrt(2)              # (|↑↓⟩ - |↓↑⟩)/√2

espin_total = ...
espin_total_cuadrado = ...
valores_propios_total = ...
singlete_por_s2 = ...

## El oscilador armónico con operadores

En el módulo `sho1d` los operadores de escalera son `RaisingOp` (el $a^\dagger$)
y `LoweringOp` (el $a$), y los estados $|n\rangle$ son `SHOKet(n)`. Un
operador de subir y uno de bajar con el mismo nombre forman un par:
$[a, a^\dagger] = 1$, y el operador número es $N = a^\dagger a$.

In [ ]:
a = LoweringOp("a")
a_dag = RaisingOp("a")
numero = NumberOp("N")

display(sp.Eq(Commutator(a, a_dag), Commutator(a, a_dag).doit(), evaluate=False))
display(sp.Eq(Commutator(numero, a_dag), Commutator(numero, a_dag).doit(), evaluate=False))

In [ ]:
n = 2   # un estado concreto para ver las reglas

display(sp.Eq(a_dag * SHOKet(n), qapply(a_dag * SHOKet(n)), evaluate=False))
display(sp.Eq(a * SHOKet(n), qapply(a * SHOKet(n)), evaluate=False))
display(sp.Eq(numero * SHOKet(n), qapply(numero * SHOKet(n)), evaluate=False))

$a^\dagger|n\rangle = \sqrt{n+1}\,|n+1\rangle$ y
$a|n\rangle = \sqrt{n}\,|n-1\rangle$.

El hamiltoniano $H = \hbar\omega\,(N + \tfrac12)$ también viene en `sho1d`, como
`Hamiltonian`. Sobre un $|n\rangle$ con $n$ **simbólico**, `qapply` da todo el
espectro de una vez:

In [ ]:
n_simbolico = sp.Symbol("n", integer=True, nonnegative=True)   # cualquier nivel: n = 0, 1, 2, ...
hamiltoniano = Hamiltonian("H")

estado_n = SHOKet(n_simbolico)

display(sp.Eq(hamiltoniano * estado_n, qapply(hamiltoniano * estado_n), evaluate=False))

Niveles separados por el mismo $\hbar\omega$, y una energía de punto cero
$\hbar\omega/2$ que no se anula ni siquiera en $n = 0$.

Las reglas de escalera bastan también para cualquier
valor esperado: la posición y el momento son combinaciones de $a$ y $a^\dagger$,

$$x = \sqrt{\frac{\hbar}{2m\omega}}\,(a + a^\dagger).$$

Calculemos $\langle n|x^2|n\rangle$ en $n = 2$ expandiendo el cuadrado y
aplicando `qapply` entre un bra y un ket:

In [ ]:
m, omega = sp.symbols("m omega", positive=True)   # masa y frecuencia: positivas

operador_x = sp.sqrt(hbar / (2 * m * omega)) * (a + a_dag)
x_cuadrada = qapply(SHOBra(n) * operador_x * operador_x * SHOKet(n))

display(sp.Eq(sp.Symbol("x2_n2"), x_cuadrada))

## TODO en clase 3

El **principio de incertidumbre** del oscilador, para el estado $|n\rangle$.

El momento es $p = i\sqrt{\dfrac{m\hbar\omega}{2}}\,(a^\dagger - a)$.

1. Define `operador_p` con esa fórmula.
2. Calcula `x2` y `p2`, los valores esperados de $x^2$ y $p^2$ en $|n\rangle$
   con $n = 3$. Usa la misma receta de arriba: `qapply(SHOBra(3) * ... * SHOKet(3))`.
3. Como $\langle x\rangle = \langle p\rangle = 0$, la incertidumbre es
   $\Delta x\,\Delta p = \sqrt{\langle x^2\rangle\langle p^2\rangle}$.
   Guárdala en `incertidumbre` y compárala con $\hbar/2$.
4. ¿Para qué $n$ se alcanza el mínimo de Heisenberg?

In [ ]:
# TODO en clase: la incertidumbre del oscilador
operador_p = ...
x2 = ...
p2 = ...
incertidumbre = ...

## Resumen

Con `quantum.spin` los operadores de momento angular actúan sobre
$|j, m\rangle$ vía `qapply`, y `represent` los vuelve matrices: para
$j = 1/2$ son las de Pauli, y `TensorProduct` junta dos espines: su espín
total separa el singlete ($S = 0$) del triplete ($S = 1$).
Con `sho1d` el oscilador se resuelve con $a$, $a^\dagger$ y $N$: el espectro
$\hbar\omega(n + \tfrac12)$ sale de una sola línea, y las reglas de escalera
bastan para obtener valores esperados sin una sola integral.

**Tarea de la semana:** [`tarea/tarea-09.ipynb`](../tarea/tarea-09.ipynb) —
unidades de una constante atómica, un conmutador, un espín en una dirección
arbitraria y el oscilador.

**Próxima semana — Semana 10:** integración de herramientas y proyecto
intermedio: cómo organizar un notebook largo en varios archivos y cómo
revisar el PR de un compañero.